<a href="https://colab.research.google.com/github/Cshoga/Intro-to-ML_Summative/blob/main/polished_student_mode_12_experiments_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Import libraries

I start by importing the libraries for data analysis, machine learning, deep learning, and plotting.

In [ ]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Libraries imported.")
print("TensorFlow version:", tf.__version__)

## 2. Load the dataset

The notebook first looks for `data.csv` in the same folder as the notebook.  
This is useful for GitHub grading because the notebook and CSV can stay together in one repository.

If the local file is not found, it uses the GitHub raw link as a backup.

In [ ]:
LOCAL_DATA_PATH = "data.csv"

# Backup link. Change this only if your GitHub repository or file location changes.
GITHUB_RAW_URL = "https://raw.githubusercontent.com/Cshoga/Intro-to-ML_Summative/main/data.csv"

if os.path.exists(LOCAL_DATA_PATH):
    DATA_PATH = LOCAL_DATA_PATH
    print("Loading local file:", DATA_PATH)
else:
    DATA_PATH = GITHUB_RAW_URL
    print("Local data.csv was not found, so I am loading from GitHub.")

df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## 3. Quick dataset check

Here I check the columns, missing values, duplicate rows, and target distribution.
This helps me understand the dataset before modelling.

In [ ]:
print("Number of rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["Target"].value_counts())

## 4. Clean column names

Some column names have spaces or special characters.  
I clean them so they are easier to use in Python.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print("Cleaned columns:")
print(df.columns.tolist()[:10])

## 5. Basic exploration

This plot shows how many students are in each target class.
I also check summary statistics for the numerical columns.

In [ ]:
plt.figure(figsize=(6, 4))
df["Target"].value_counts().plot(kind="bar")
plt.title("Distribution of Student Academic Outcomes")
plt.xlabel("Outcome")
plt.ylabel("Number of students")
plt.xticks(rotation=0)
plt.show()

df.describe().T.head(15)

## 6. Simple feature engineering

I created a few extra features from the semester information.  
These features are simple, but they may help the models understand student academic progress better.

In [ ]:
df_fe = df.copy()

df_fe["Total_approved_units"] = (
    df_fe["Curricular_units_1st_sem_approved"] +
    df_fe["Curricular_units_2nd_sem_approved"]
)

df_fe["Average_semester_grade"] = (
    df_fe["Curricular_units_1st_sem_grade"] +
    df_fe["Curricular_units_2nd_sem_grade"]
) / 2

df_fe["Total_evaluations"] = (
    df_fe["Curricular_units_1st_sem_evaluations"] +
    df_fe["Curricular_units_2nd_sem_evaluations"]
)

df_fe["Total_enrolled_units"] = (
    df_fe["Curricular_units_1st_sem_enrolled"] +
    df_fe["Curricular_units_2nd_sem_enrolled"]
)

print("Shape after feature engineering:", df_fe.shape)

df_fe[
    ["Total_approved_units", "Average_semester_grade",
     "Total_evaluations", "Total_enrolled_units", "Target"]
].head()

## 7. Prepare features and target

The target is text, so I convert it into numbers:

- Dropout = 0
- Enrolled = 1
- Graduate = 2

In [ ]:
target_map = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
label_map = {0: "Dropout", 1: "Enrolled", 2: "Graduate"}

df_fe["Target_encoded"] = df_fe["Target"].map(target_map)

X = df_fe.drop(columns=["Target", "Target_encoded"])
y = df_fe["Target_encoded"]

num_classes = len(np.unique(y))

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nEncoded target distribution:")
print(y.value_counts().rename(index=label_map))

## 8. Train, validation, and test split

I split the data into:

- training data for learning
- validation data for checking deep learning performance during training
- test data for final evaluation

I use stratification so that the class distribution stays balanced in each split.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

## 9. Scaling for neural networks

Traditional models that need scaling use a Scikit-learn pipeline.  
For deep learning, I scale the data here before creating TensorFlow datasets.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

input_dim = X_train_scaled.shape[1]

print("Scaling completed.")
print("Input dimension for neural networks:", input_dim)
print("Number of classes:", num_classes)

## 10. Evaluation helper functions

These functions avoid repeating the same evaluation code for every experiment.  
I use accuracy, weighted precision, weighted recall, and weighted F1-score because this is a multiclass problem.

In [ ]:
results = []

def evaluate_model(name, model_type, y_true, y_pred, y_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    results.append({
        "Experiment": name,
        "Type": model_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    print(f"===== {name} =====")
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=[label_map[i] for i in range(num_classes)],
        zero_division=0
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        display_labels=[label_map[i] for i in range(num_classes)],
        xticks_rotation=45
    )
    plt.title("Confusion Matrix - " + name)
    plt.show()


def plot_multiclass_roc(y_true, y_proba, title):
    y_bin = label_binarize(y_true, classes=list(range(num_classes)))

    plt.figure(figsize=(7, 5))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{label_map[i]} AUC = {roc_auc:.2f}")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()


def plot_sklearn_learning_curve(model, X_data, y_data, title):
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X_data,
        y_data,
        cv=3,
        scoring="f1_weighted",
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=-1
    )

    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training score")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation score")
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Weighted F1-score")
    plt.legend()
    plt.show()


def student_note():
    print("\nMy note: I will compare the training curve and validation curve.")
    print("If the training score is much higher than validation, the model may be overfitting.")
    print("If both scores are low, the model may be underfitting.")

# Traditional Machine Learning Experiments

I first test six Scikit-learn models.  
This gives me a baseline before comparing with deep learning.

## Experiment 1: Logistic Regression


Logistic Regression is a simple baseline model.  
It is useful because it is easy to interpret and works well for many classification problems.

In [ ]:
ml1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED))
])

ml1.fit(X_train_full, y_train_full)

pred = ml1.predict(X_test)
proba = ml1.predict_proba(X_test)

evaluate_model("E1 Logistic Regression", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E1 Logistic Regression")
plot_sklearn_learning_curve(ml1, X_train_full, y_train_full, "Learning Curve - E1 Logistic Regression")
student_note()

## Experiment 2: Decision Tree


Decision Tree is easy to understand because it makes decisions using feature splits.  
I limit the depth to reduce overfitting.

In [ ]:
ml2 = DecisionTreeClassifier(max_depth=5, random_state=SEED)

ml2.fit(X_train_full, y_train_full)

pred = ml2.predict(X_test)
proba = ml2.predict_proba(X_test)

evaluate_model("E2 Decision Tree", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E2 Decision Tree")
plot_sklearn_learning_curve(ml2, X_train_full, y_train_full, "Learning Curve - E2 Decision Tree")
student_note()

## Experiment 3: Random Forest


Random Forest combines many decision trees.  
I used class weighting because the target classes are not perfectly balanced.

In [ ]:
ml3 = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    random_state=SEED,
    class_weight="balanced"
)

ml3.fit(X_train_full, y_train_full)

pred = ml3.predict(X_test)
proba = ml3.predict_proba(X_test)

evaluate_model("E3 Random Forest", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E3 Random Forest")
plot_sklearn_learning_curve(ml3, X_train_full, y_train_full, "Learning Curve - E3 Random Forest")
student_note()

## Experiment 4: Gradient Boosting


Gradient Boosting builds trees one after another.  
Each new tree tries to correct previous mistakes.

In [ ]:
ml4 = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=3,
    random_state=SEED
)

ml4.fit(X_train_full, y_train_full)

pred = ml4.predict(X_test)
proba = ml4.predict_proba(X_test)

evaluate_model("E4 Gradient Boosting", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E4 Gradient Boosting")
plot_sklearn_learning_curve(ml4, X_train_full, y_train_full, "Learning Curve - E4 Gradient Boosting")
student_note()

## Experiment 5: Support Vector Machine


SVM can create a non-linear boundary using the RBF kernel.  
I included probability=True so I can plot ROC curves.

In [ ]:
ml5 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED))
])

ml5.fit(X_train_full, y_train_full)

pred = ml5.predict(X_test)
proba = ml5.predict_proba(X_test)

evaluate_model("E5 SVM RBF", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E5 SVM RBF")
plot_sklearn_learning_curve(ml5, X_train_full, y_train_full, "Learning Curve - E5 SVM RBF")
student_note()

## Experiment 6: K-Nearest Neighbors


KNN predicts using the nearest examples in the training data.  
This model is simple, but it can be sensitive to scaling.

In [ ]:
ml6 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

ml6.fit(X_train_full, y_train_full)

pred = ml6.predict(X_test)
proba = ml6.predict_proba(X_test)

evaluate_model("E6 KNN", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E6 KNN")
plot_sklearn_learning_curve(ml6, X_train_full, y_train_full, "Learning Curve - E6 KNN")
student_note()

# Deep Learning Preparation

For the deep learning part, I use the TensorFlow `tf.data` API.  
This satisfies the assignment requirement and also makes batching and prefetching easier.

In [ ]:
BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train_scaled.astype("float32"), y_train.values.astype("int32"))
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (X_val_scaled.astype("float32"), y_val.values.astype("int32"))
)

test_ds = tf.data.Dataset.from_tensor_slices(
    (X_test_scaled.astype("float32"), y_test.values.astype("int32"))
)

train_ds = train_ds.shuffle(
    buffer_size=len(X_train_scaled),
    seed=SEED
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("tf.data datasets are ready.")
print("Input dimension:", input_dim)

## Deep learning helper functions

These functions compile the neural network, plot the learning curves, and evaluate the model on the test set.

In [ ]:
def compile_model(model, learning_rate=0.001):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def plot_dl_history(history, title):
    plt.figure(figsize=(7, 5))
    plt.plot(history.history["accuracy"], label="Training accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation accuracy")
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


def evaluate_dl_model(name, model):
    y_proba = model.predict(X_test_scaled.astype("float32"))
    y_pred = np.argmax(y_proba, axis=1)

    evaluate_model(name, "Deep Learning", y_test, y_pred, y_proba)
    plot_multiclass_roc(y_test, y_proba, "ROC Curve - " + name)


def dl_student_note():
    print("\nMy note: In the accuracy and loss curves, I check if validation performance improves.")
    print("If validation loss increases while training loss decreases, the neural network may be overfitting.")

## Experiment 7: Sequential API - Small Neural Network


This is my first deep learning baseline.  
It uses the Keras Sequential API with one hidden layer.

In [ ]:
def build_seq_small():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.001)

dl1 = build_seq_small()

print("Architecture summary for E7:")
dl1.summary()

history1 = dl1.fit(train_ds, validation_data=val_ds, epochs=20, verbose=1)

plot_dl_history(history1, "E7 Sequential Small NN")
evaluate_dl_model("E7 Sequential Small NN", dl1)
dl_student_note()

## Experiment 8: Sequential API - Medium Neural Network with Dropout


This model is a bit deeper and uses dropout.  
Dropout helps reduce overfitting by randomly turning off some neurons during training.

In [ ]:
def build_seq_medium_dropout():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.001)

dl2 = build_seq_medium_dropout()

print("Architecture summary for E8:")
dl2.summary()

history2 = dl2.fit(train_ds, validation_data=val_ds, epochs=25, verbose=1)

plot_dl_history(history2, "E8 Sequential Medium NN with Dropout")
evaluate_dl_model("E8 Sequential Medium NN with Dropout", dl2)
dl_student_note()

## Experiment 9: Sequential API - Lower Learning Rate


This experiment keeps a similar architecture but lowers the learning rate.  
A smaller learning rate can make training more stable, although it may train more slowly.

In [ ]:
def build_seq_low_lr():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.0005)

dl3 = build_seq_low_lr()

print("Architecture summary for E9:")
dl3.summary()

history3 = dl3.fit(train_ds, validation_data=val_ds, epochs=25, verbose=1)

plot_dl_history(history3, "E9 Sequential Lower Learning Rate")
evaluate_dl_model("E9 Sequential Lower Learning Rate", dl3)
dl_student_note()

## Experiment 10: Functional API - Two Hidden Layers


This experiment uses the Keras Functional API.  
The architecture is still simple, but Functional API gives more flexibility for complex models.

In [ ]:
def build_functional_basic():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

dl4 = build_functional_basic()

print("Architecture summary for E10:")
dl4.summary()

history4 = dl4.fit(train_ds, validation_data=val_ds, epochs=25, verbose=1)

plot_dl_history(history4, "E10 Functional Basic NN")
evaluate_dl_model("E10 Functional Basic NN", dl4)
dl_student_note()

## Experiment 11: Functional API - Batch Normalization


This model adds Batch Normalization.  
Batch Normalization can help make training more stable.

In [ ]:
def build_functional_batchnorm():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

dl5 = build_functional_batchnorm()

print("Architecture summary for E11:")
dl5.summary()

history5 = dl5.fit(train_ds, validation_data=val_ds, epochs=30, verbose=1)

plot_dl_history(history5, "E11 Functional NN with BatchNorm")
evaluate_dl_model("E11 Functional NN with BatchNorm", dl5)
dl_student_note()

## Experiment 12: Functional API - Early Stopping


This final deep learning experiment uses early stopping.  
Early stopping stops training when validation loss no longer improves.

In [ ]:
def build_functional_earlystop():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

dl6 = build_functional_earlystop()

print("Architecture summary for E12:")
dl6.summary()

history6 = dl6.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

plot_dl_history(history6, "E12 Functional NN with Early Stopping")
evaluate_dl_model("E12 Functional NN with Early Stopping", dl6)
dl_student_note()

# Final Results Comparison

This table compares all 12 experiments.  
I sort the models by weighted F1-score because the project is multiclass and the classes are not exactly equal in size.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)

results_df

## Final comparison plot

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Experiment"], results_df["F1-score"])
plt.title("Final Model Comparison by Weighted F1-score")
plt.xlabel("Experiment")
plt.ylabel("Weighted F1-score")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## My short conclusion notes

After running all experiments, I will use the final table to identify the best model.  
In my report, I will not only choose the model with the highest F1-score, but also explain the learning curves, confusion matrix, and possible mistakes.

Important limitations I should mention:

The dataset is useful for academic outcome prediction, but it may not represent every education system. Some important personal factors, such as motivation, family support, and mental health, may not be fully captured in the data. Also, the model should not be used to label students negatively. It should only support early intervention and student support.

## Simple report and video reminder

For the written report, I will discuss the problem, dataset, preprocessing, experiments, results, limitations, and conclusion.  
For the video, I will show the notebook, explain the best model, and describe what I learned from comparing machine learning with deep learning.